# Profilage DVF 2022 — Etape 2c : distributions et anomalies

Les etapes precedentes ont decouvert la structure du fichier (etape 1),
visualise les constats principaux (etape 2a) et explore les doublons (etape 2b).

Ce notebook examine les **distributions** des variables cles et repere
les **anomalies** (valeurs extremes, cas atypiques) :

1. Repartition temporelle (par mois)
2. Repartition geographique (par departement)
3. Distributions des surfaces (batie et terrain)
4. Valeurs extremes de la valeur fonciere
5. Transactions a 0 euro

---

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "matplotlib", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [2]:
import duckdb
import os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.family": "sans-serif", "font.size": 10})

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Repartition temporelle : par mois

Le fichier couvre le millesime 2022. Combien de transactions
ont ete enregistrees chaque mois ? La repartition est-elle
reguliere ou certains mois concentrent-ils un volume
anormalement eleve ou faible ?

In [3]:
par_mois = con.execute(f"""
    SELECT
        month("Date mutation") AS mois,
        count(*) AS nb_lignes,
        round(100.0 * count(*) / {nb_lignes}, 2) AS pct
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print(f"{'Mois':<8} {'Nb lignes':>12} {'Pct':>8}")
print("-" * 30)
for mois, nb, pct in par_mois:
    print(f"  {mois:<8} {nb:>10,} {pct:>7.2f} %".replace(",", " "))

# Graphique
mois_noms = ["Jan","Fev","Mar","Avr","Mai","Jun","Jul","Aou","Sep","Oct","Nov","Dec"]
vals_mois = [nb for _, nb, _ in par_mois]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(mois_noms, vals_mois, color="#2E75B6", edgecolor="white")
ax.set_ylabel("Nombre de lignes")
ax.set_title("Repartition temporelle des transactions (2022)")
for i, v in enumerate(vals_mois):
    ax.text(i, v + 5000, f"{v:,}".replace(",", " "), ha="center", fontsize=7)
plt.tight_layout()
fig.savefig(SORTIE / "fig4_repartition_mensuelle.png", dpi=150, bbox_inches="tight")
print(f"\nGraphique enregistre : fig4_repartition_mensuelle.png")
plt.show()

Mois        Nb lignes      Pct
------------------------------
  1           323 060    7.00 %
  2           334 988    7.25 %
  3           410 709    8.89 %
  4           365 804    7.92 %
  5           380 910    8.25 %
  6           445 511    9.65 %
  7           449 108    9.73 %
  8           302 538    6.55 %
  9           417 144    9.03 %
  10          359 369    7.78 %
  11          335 975    7.28 %
  12          492 474   10.67 %

Graphique enregistre : fig4_repartition_mensuelle.png


[chemin temporaire]:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cellule 4 — Plage de dates

Verifier que toutes les transactions sont bien dans l'annee 2022.
Des dates hors plage signaleraient une anomalie de coherence.

In [4]:
plage = con.execute(f"""
    SELECT
        min("Date mutation") AS premiere_date,
        max("Date mutation") AS derniere_date,
        count(DISTINCT "Date mutation") AS nb_dates_distinctes,
        count(CASE WHEN year("Date mutation") != 2022 THEN 1 END) AS hors_2022
    FROM '{pq}'
""").fetchone()

print("Plage de dates")
print("=" * 40)
print(f"  Premiere date      : {plage[0]}")
print(f"  Derniere date      : {plage[1]}")
print(f"  Dates distinctes   : {plage[2]}")
print(f"  Lignes hors 2022   : {plage[3]}")

Plage de dates
  Premiere date      : 2022-01-01
  Derniere date      : 2022-12-31
  Dates distinctes   : 359
  Lignes hors 2022   : 0


## Cellule 5 — Repartition geographique : top 20 departements

Quels departements ont le plus de transactions ?
Observer si la repartition suit la densite de population
ou si des ecarts importants existent.

In [5]:
par_dept = con.execute(f"""
    SELECT
        "Code departement",
        count(*) AS nb_lignes,
        round(100.0 * count(*) / {nb_lignes}, 2) AS pct,
        round(median("Valeur fonciere"), 0) AS mediane_prix
    FROM '{pq}'
    WHERE "Valeur fonciere" IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 20
""").fetchall()

print(f"{'Dept':<6} {'Nb lignes':>12} {'Pct':>8} {'Mediane prix':>14}")
print("-" * 45)
for dept, nb, pct, med in par_dept:
    print(f"  {dept:<6} {nb:>10,} {pct:>7.2f} % {med:>12,.0f}".replace(",", " "))

# Graphique
depts = [d[0] for d in par_dept]
nb_vals = [d[1] for d in par_dept]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(depts)), nb_vals, color="#2E75B6")
ax.set_xticks(range(len(depts)))
ax.set_xticklabels(depts, fontsize=8)
ax.set_xlabel("Code departement")
ax.set_ylabel("Nombre de lignes")
ax.set_title("Top 20 des departements par volume de transactions")
plt.tight_layout()
fig.savefig(SORTIE / "fig5_top20_departements.png", dpi=150, bbox_inches="tight")
print(f"\nGraphique enregistre : fig5_top20_departements.png")
plt.show()

Dept      Nb lignes      Pct   Mediane prix
---------------------------------------------
  59        122 024    2.64 %      173 000
  33        119 152    2.58 %      250 000
  13        108 384    2.35 %      265 000
  83        106 502    2.31 %      283 000
  69        106 254    2.30 %      286 000
  06        105 870    2.29 %      274 787
  44         99 939    2.16 %      206 000
  31         96 792    2.10 %      208 000
  34         96 588    2.09 %      181 802
  75         95 826    2.08 %      530 675
  56         92 544    2.00 %      287 000
  77         84 087    1.82 %      235 000
  74         81 618    1.77 %      283 860
  92         79 912    1.73 %      458 000
  35         77 698    1.68 %      181 900
  38         75 504    1.64 %      184 500
  29         74 076    1.60 %      140 000
  17         70 150    1.52 %      169 000
  62         70 134    1.52 %      145 000
  22         70 032    1.52 %      150 000

Graphique enregistre : fig5_top20_departements.pn

[chemin temporaire]:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cellule 6 — Distribution de la surface reelle batie

La surface batie est l'une des variables explicatives les plus
importantes pour predire le prix d'un bien.

On examine sa distribution pour reperer :

- La forme generale (symetrique, asymetrique ?)
- Les valeurs extremes (surfaces de 0, ou de plusieurs milliers de m2)
- Le centre de la distribution (mediane)

In [6]:
stats_surface = con.execute(f"""
    SELECT
        count("Surface reelle bati") AS nb_non_null,
        min("Surface reelle bati") AS minimum,
        approx_quantile("Surface reelle bati", 0.01) AS P1,
        approx_quantile("Surface reelle bati", 0.05) AS P5,
        approx_quantile("Surface reelle bati", 0.25) AS Q1,
        median("Surface reelle bati") AS mediane,
        approx_quantile("Surface reelle bati", 0.75) AS Q3,
        approx_quantile("Surface reelle bati", 0.95) AS P95,
        approx_quantile("Surface reelle bati", 0.99) AS P99,
        max("Surface reelle bati") AS maximum,
        round(avg("Surface reelle bati"), 0) AS moyenne,
        count(CASE WHEN "Surface reelle bati" = 0 THEN 1 END) AS nb_zero
    FROM '{pq}'
    WHERE "Surface reelle bati" IS NOT NULL
""").fetchone()

labels_s = ["Nb non null", "Minimum", "P1", "P5", "Q1", "Mediane",
            "Q3", "P95", "P99", "Maximum", "Moyenne", "Nb a 0 m2"]

print("Statistiques — Surface reelle batie (m2)")
print("=" * 40)
for label, val in zip(labels_s, stats_surface):
    if isinstance(val, (int, float)):
        print(f"  {label:<15} {val:>12,.0f}".replace(",", " "))
    else:
        print(f"  {label:<15} {val}")

Statistiques — Surface reelle batie (m2)
  Nb non null        2 738 208
  Minimum                    0
  P1                         0
  P5                         0
  Q1                         0
  Mediane                   31
  Q3                        81
  P95                      154
  P99                      385
  Maximum              290 000
  Moyenne                   65
  Nb a 0 m2          1 208 607


In [7]:
# Histogramme de la surface batie (tronque a 500 m2 pour lisibilite)
surf_vals = con.execute(f"""
    SELECT "Surface reelle bati"
    FROM '{pq}'
    WHERE "Surface reelle bati" IS NOT NULL
      AND "Surface reelle bati" > 0
      AND "Surface reelle bati" <= 500
""").fetchnumpy()["Surface reelle bati"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(surf_vals, bins=100, color="#2E75B6", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Surface reelle batie (m2)")
ax.set_ylabel("Nombre de biens")
ax.set_title("Distribution de la surface batie (0-500 m2)")
plt.tight_layout()
fig.savefig(SORTIE / "fig6_distribution_surface_batie.png", dpi=150, bbox_inches="tight")
print(f"Graphique enregistre : fig6_distribution_surface_batie.png")
plt.show()

Graphique enregistre : fig6_distribution_surface_batie.png


[chemin temporaire]:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cellule 7 — Distribution de la surface terrain

Meme analyse pour la surface terrain. Cette variable est
structurellement differente de la surface batie : elle peut
atteindre des valeurs beaucoup plus elevees (terrains agricoles).

In [8]:
stats_terrain = con.execute(f"""
    SELECT
        count("Surface terrain") AS nb_non_null,
        min("Surface terrain") AS minimum,
        approx_quantile("Surface terrain", 0.05) AS P5,
        approx_quantile("Surface terrain", 0.25) AS Q1,
        median("Surface terrain") AS mediane,
        approx_quantile("Surface terrain", 0.75) AS Q3,
        approx_quantile("Surface terrain", 0.95) AS P95,
        approx_quantile("Surface terrain", 0.99) AS P99,
        max("Surface terrain") AS maximum,
        round(avg("Surface terrain"), 0) AS moyenne,
        count(CASE WHEN "Surface terrain" = 0 THEN 1 END) AS nb_zero
    FROM '{pq}'
    WHERE "Surface terrain" IS NOT NULL
""").fetchone()

labels_t = ["Nb non null", "Minimum", "P5", "Q1", "Mediane",
            "Q3", "P95", "P99", "Maximum", "Moyenne", "Nb a 0 m2"]

print("Statistiques — Surface terrain (m2)")
print("=" * 40)
for label, val in zip(labels_t, stats_terrain):
    if isinstance(val, (int, float)):
        print(f"  {label:<15} {val:>15,.0f}".replace(",", " "))
    else:
        print(f"  {label:<15} {val}")

Statistiques — Surface terrain (m2)
  Nb non null           3 099 199
  Minimum                       0
  P5                           34
  Q1                          246
  Mediane                     609
  Q3                        1 674
  P95                      11 372
  P99                      36 751
  Maximum               4 625 500
  Moyenne                   2 795
  Nb a 0 m2                    76


## Cellule 8 — Valeurs extremes de la valeur fonciere

L'etape 1 a montre que la valeur fonciere va de 0 a plus d'un milliard.
Quels sont les 10 prix les plus eleves et les 10 plus bas (hors zero) ?
Les cas extremes sont-ils des transactions plausibles
ou des anomalies ?

In [9]:
# Top 10 plus chers
top10 = con.execute(f"""
    SELECT "Date mutation", "Valeur fonciere", "Commune",
           "Code departement", "Type local", "Surface reelle bati",
           "Nature mutation"
    FROM '{pq}'
    WHERE "Valeur fonciere" IS NOT NULL
    ORDER BY "Valeur fonciere" DESC
    LIMIT 10
""").fetchdf()

print("10 transactions les plus cheres :")
display(top10)

10 transactions les plus cheres :


,Date mutation,Valeur fonciere,Commune,Code departement,Type local,Surface reelle bati,Nature mutation
0,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
1,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
2,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
3,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
4,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
5,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
6,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
7,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
8,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente
9,2022-12-19,1003401470,PARIS 13,75,None,<NA>,Vente


In [ ]:
# 10 prix les plus bas (> 0)
bottom10 = con.execute(f"""
    SELECT "Date mutation", "Valeur fonciere", "Commune",
           "Code departement", "Type local", "Surface reelle bati",
           "Nature mutation"
    FROM '{pq}'
    WHERE "Valeur fonciere" IS NOT NULL AND "Valeur fonciere" > 0
    ORDER BY "Valeur fonciere" ASC
    LIMIT 10
""").fetchdf()

print("10 transactions les moins cheres (> 0 euro) :")
display(bottom10)

## Cellule 9 — Transactions a 0 euro

L'etape 1 a identifie 33 transactions a 0 euro.
Observer leur profil pour determiner s'il s'agit de donations,
de cessions administratives, ou d'erreurs de saisie.

In [10]:
zeros = con.execute(f"""
    SELECT "Date mutation", "Nature mutation", "Commune",
           "Code departement", "Type local", "Surface reelle bati",
           "Surface terrain"
    FROM '{pq}'
    WHERE "Valeur fonciere" = 0
    ORDER BY "Date mutation"
""").fetchdf()

print(f"Transactions a 0 euro : {len(zeros)} lignes")
print()

# Repartition par nature de mutation
print("Par nature de mutation :")
nat_zeros = zeros.groupby("Nature mutation").size().sort_values(ascending=False)
for nat, nb in nat_zeros.items():
    print(f"  {nat:<40} {nb}")

print()
print("Apercu des premieres lignes :")
display(zeros.head(10))

Transactions a 0 euro : 33 lignes

Par nature de mutation :
  Vente                                    25
  Echange                                  8

Apercu des premieres lignes :


,Date mutation,Nature mutation,Commune,Code departement,Type local,Surface reelle bati,Surface terrain
0,2022-01-31,Vente,MENNECY,91,Local industriel. commercial ou assimilé,1128,2750
1,2022-02-10,Vente,LYON 6EME,69,Local industriel. commercial ou assimilé,640,<NA>
2,2022-03-22,Vente,NOEUX LES MINES,62,Local industriel. commercial ou assimilé,396,4492
3,2022-03-22,Vente,NOEUX LES MINES,62,Local industriel. commercial ou assimilé,855,1659
4,2022-03-31,Vente,BAZET,65,None,<NA>,71
5,2022-03-31,Vente,BAZET,65,None,<NA>,8
6,2022-04-08,Vente,FONTENAY LE FLEURY,78,Dépendance,0,<NA>
7,2022-04-08,Vente,FONTENAY LE FLEURY,78,Appartement,43,<NA>
8,2022-05-17,Vente,IVRY SUR SEINE,94,None,<NA>,<NA>
9,2022-05-17,Vente,IVRY SUR SEINE,94,None,<NA>,<NA>


## Cellule 10 — Valeur fonciere par type de bien

Le prix median varie-t-il selon le type de bien ?
Ce croisement permet de verifier si les types de biens
correspondent a des gammes de prix differentes,
ce qui serait un signe de coherence interne du jeu de donnees.

In [11]:
prix_type = con.execute(f"""
    SELECT
        CASE
            WHEN "Type local" IS NULL THEN '(vide)'
            ELSE "Type local"
        END AS type_local,
        count(*) AS nb,
        round(median("Valeur fonciere"), 0) AS mediane,
        round(avg("Valeur fonciere"), 0) AS moyenne,
        approx_quantile("Valeur fonciere", 0.25) AS Q1,
        approx_quantile("Valeur fonciere", 0.75) AS Q3
    FROM '{pq}'
    WHERE "Valeur fonciere" IS NOT NULL AND "Valeur fonciere" > 0
    GROUP BY 1
    ORDER BY 3 DESC
""").fetchdf()

print("Valeur fonciere par type de bien")
display(prix_type)

Valeur fonciere par type de bien


,type_local,nb,mediane,moyenne,Q1,Q3
0,Local industriel. commercial ou assimilé,141224,280000.0,2625720.0,123982,753498
1,Maison,755030,216000.0,763900.0,131194,343109
2,Dépendance,1200891,200000.0,4642320.0,115291,354060
3,Appartement,636512,196795.0,8374307.0,119353,362195
4,(vide),1852758,110000.0,598039.0,20184,261857


## Cellule 11 — Transactions sans valeur fonciere

L'etape 1 a identifie 31 142 lignes sans valeur fonciere (0,67 %).
Observer leur profil pour determiner si elles sont concentrees
sur un type de bien, un departement ou une nature de mutation.

In [12]:
sans_prix = con.execute(f"""
    SELECT
        CASE
            WHEN "Type local" IS NULL THEN '(vide)'
            ELSE "Type local"
        END AS type_local,
        "Nature mutation",
        count(*) AS nb,
        round(100.0 * count(*) / 31142, 1) AS pct
    FROM '{pq}'
    WHERE "Valeur fonciere" IS NULL
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""").fetchdf()

print("Profil des lignes sans valeur fonciere (top 10 combinaisons) :")
display(sans_prix)

Profil des lignes sans valeur fonciere (top 10 combinaisons) :


,type_local,Nature mutation,nb,pct
0,(vide),Vente,21187,68.0
1,Dépendance,Vente,2485,8.0
2,Appartement,Vente,2329,7.5
3,(vide),Echange,2285,7.3
4,Local industriel. commercial ou assimilé,Vente,1272,4.1
5,Maison,Vente,934,3.0
6,(vide),Expropriation,306,1.0
7,(vide),Vente en l'état futur d'achèvement,124,0.4
8,(vide),Vente terrain à bâtir,40,0.1
9,Dépendance,Expropriation,38,0.1


## Cellule 12 — Synthese de l'etape 2c

In [13]:
print("Synthese — Distributions et anomalies DVF 2022")
print("=" * 55)
print(f"  Plage de dates    : {plage[0]} a {plage[1]}")
print(f"  Dates distinctes  : {plage[2]}")
print(f"  Hors 2022         : {plage[3]} lignes")
print()
print("  Surface batie :")
print(f"    Mediane         : {stats_surface[5]:,.0f} m2".replace(",", " "))
print(f"    Maximum         : {stats_surface[9]:,.0f} m2".replace(",", " "))
print(f"    Nb a 0 m2       : {stats_surface[11]:,.0f}".replace(",", " "))
print()
print("  Surface terrain :")
print(f"    Mediane         : {stats_terrain[4]:,.0f} m2".replace(",", " "))
print(f"    Maximum         : {stats_terrain[8]:,.0f} m2".replace(",", " "))
print(f"    Nb a 0 m2       : {stats_terrain[10]:,.0f}".replace(",", " "))
print()
print(f"  Transactions a 0 euro : {len(zeros)}")

Synthese — Distributions et anomalies DVF 2022
  Plage de dates    : 2022-01-01 a 2022-12-31
  Dates distinctes  : 359
  Hors 2022         : 0 lignes

  Surface batie :
    Mediane         : 31 m2
    Maximum         : 290 000 m2
    Nb a 0 m2       : 1 208 607

  Surface terrain :
    Mediane         : 609 m2
    Maximum         : 4 625 500 m2
    Nb a 0 m2       : 76

  Transactions a 0 euro : 33


## Cellule 13 — Fermeture

In [14]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
